# Geçen yılın aynı dönemi (SPLY) denemesi

**Hipotez:** Test dönemi Nisan-Temmuz 2026. Bir trafonun geçen yıl aynı tarihlerde
(Nisan-Temmuz 2025) ne tükettiği, bu yıl ne tüketeceğinin güçlü bir göstergesi olmalı.
Turizm bölgesindeki trafo her yaz artar, sanayi trafosu ağustosta düşer, sulama trafosu
haziran-eylül arası çalışır. Bu yıllık tekrarı modele hiç söylemiyoruz.

Burada iki şeyi ölçüyoruz:
1. Hangi pencere en iyisi? (tam aynı gün mü, o haftanın ortalaması mı, ±15 gün mü)
2. Bu bilgi, elimizdeki mevcut özelliklerin ötesinde bir şey katıyor mu?

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

RAW = '../data/raw/'
train = pd.read_csv(RAW + 'train.csv')
train['tarih'] = pd.to_datetime(train['tarih'])

# Fiziksel olarak imkânsız değerleri trafo medyanıyla düzelt (EDA'daki 23 problemli trafo)
train['asiri_oran'] = train['tuketim'] / (train['guc'] * 24)
train['tuketim_duzeltilmis'] = train['tuketim']
for t in train.loc[train['asiri_oran'] > 2, 'tanim'].unique():
    medyan = train.loc[(train['tanim'] == t) & (train['asiri_oran'] <= 2), 'tuketim'].median()
    train.loc[(train['tanim'] == t) & (train['asiri_oran'] > 2), 'tuketim_duzeltilmis'] = medyan

lok = train['lokasyon'].str.split('>', expand=True)
train['il'], train['bolge'] = lok[0], lok[1]
train['ilce'] = lok[2].fillna('YOK')

train['ay'] = train['tarih'].dt.month
train['gun'] = train['tarih'].dt.day
train['haftanin_gunu'] = train['tarih'].dt.dayofweek
train['hafta_sonu'] = (train['haftanin_gunu'] >= 5).astype(int)
train['yilin_gunu'] = train['tarih'].dt.dayofyear
train['guc_bucket'] = pd.cut(train['guc'], bins=[0,100,250,500,1000,2000,5000,np.inf])
train['guc_bucket_str'] = train['guc_bucket'].astype(str)
train['log_t'] = np.log1p(train['tuketim_duzeltilmis'])

KATEGORIK = ['il','bolge','ilce','guc_bucket_str']
print(train.shape)

(1226237, 18)


## SPLY özelliklerini kur

Dört farklı pencere deniyoruz: tam aynı gün (w=0), o haftanın ortalaması (±3),
iki haftalık (±7), bir aylık (±15). Pencere genişledikçe gürültü azalır ama
gün-spesifik bilgi kaybolur; hangisinin daha iyi olduğu ölçülecek.

In [2]:
# Trafo x tarih matrisi kurup her pencere için ortalamayı önceden hesaplıyoruz
piv = train.pivot_table(index='tarih', columns='tanim', values='log_t')
piv = piv.reindex(pd.date_range(piv.index.min(), piv.index.max()))

PENCERELER = [0, 3, 7, 15]
ROLL = {w: piv.rolling(2*w + 1, center=True, min_periods=1).mean() for w in PENCERELER}

trafo_yil_ort = train.groupby('tanim')['log_t'].mean()


def sply_ekle(df, origin):
    df = df.copy()
    gecen_yil = df['tarih'] - pd.Timedelta(days=365)
    for w in PENCERELER:
        R = ROLL[w]
        R = R[R.index < origin]          # origin sonrası hiçbir veri kullanılmasın
        idx = pd.MultiIndex.from_arrays([gecen_yil, df['tanim']])
        df[f'sply_w{w}'] = R.stack(future_stack=True).reindex(idx).values
    # Seviyeden bağımsız hâli: trafonun kendi yıllık ortalamasından sapma
    df['sply_sapma'] = df['sply_w7'] - df['tanim'].map(trafo_yil_ort)
    df['sply_var'] = df['sply_w7'].notna().astype(int)
    return df

In [3]:
# Diğer özellikler (02_model.ipynb ile aynı mantık, buraya kompakt hâlde alındı)
def trafo_profili(origin):
    g = train[train['tarih'] < origin]
    p = g.groupby('tanim').agg(
        trafo_ortalama=('tuketim_duzeltilmis','mean'), trafo_medyan=('tuketim_duzeltilmis','median'),
        trafo_gun_sayisi=('log_t','count'), log_ortalama=('log_t','mean'),
        log_medyan=('log_t','median'), log_std=('log_t','std'),
        sifir_orani=('tuketim_duzeltilmis', lambda x: (x == 0).mean())).reset_index()
    for w in [30, 90]:
        s = g[g['tarih'] >= origin - pd.Timedelta(days=w)]
        o = s.groupby('tanim').agg(**{
            f'son{w}_log_ort': ('log_t','mean'),
            f'son{w}_sifir': ('tuketim_duzeltilmis', lambda x: (x == 0).mean())}).reset_index()
        p = p.merge(o, on='tanim', how='left')
    p['log_trend'] = p['son30_log_ort'] - p['son90_log_ort']
    return p


def gun_profili(origin):
    g = train[train['tarih'] < origin][['tanim','haftanin_gunu','tuketim_duzeltilmis']].copy()
    o = g.groupby('tanim')['tuketim_duzeltilmis'].mean().rename('_o')
    g = g.merge(o, on='tanim', how='left')
    g['oran'] = g['tuketim_duzeltilmis'] / g['_o'].replace(0, np.nan)
    return g.groupby(['tanim','haftanin_gunu'])['oran'].mean().rename('trafo_gun_orani').reset_index()


def veri_seti(hedef, origin, k=30):
    g = train[train['tarih'] < origin]
    genel = g['log_t'].mean()
    grup = g.groupby(['il','bolge','guc_bucket'], observed=True)['log_t'].agg(['sum','count']).reset_index()
    grup['grup_log_ort'] = (grup['sum'] + k*genel) / (grup['count'] + k)
    ilguc = g.groupby(['il','guc_bucket'], observed=True)['log_t'].mean().rename('ilguc_log_ort').reset_index()

    df = hedef.merge(grup[['il','bolge','guc_bucket','grup_log_ort']], on=['il','bolge','guc_bucket'], how='left')
    df = df.merge(ilguc, on=['il','guc_bucket'], how='left')
    df['grup_log_ort'] = df['grup_log_ort'].fillna(df['ilguc_log_ort']).fillna(genel)
    df = df.merge(trafo_profili(origin), on='tanim', how='left')
    df = df.merge(gun_profili(origin), on=['tanim','haftanin_gunu'], how='left')
    return sply_ekle(df, origin)


TEMEL = ['guc','ay','gun','haftanin_gunu','hafta_sonu','yilin_gunu','il','bolge','ilce','guc_bucket_str',
         'grup_log_ort','trafo_ortalama','trafo_medyan','trafo_gun_sayisi','log_ortalama','log_medyan',
         'log_std','sifir_orani','son30_log_ort','son90_log_ort','son30_sifir','son90_sifir',
         'log_trend','trafo_gun_orani']

def rmsle(gercek, tahmin):
    return np.sqrt(np.mean((np.log1p(tahmin) - np.log1p(gercek))**2))

## Kapsama sorunu

Veri Ocak 2025'te başlıyor, yani "geçen yıl" karşılığı ancak 2026 tarihleri için var.
Eğitim hedefimiz normalde 2025 içindeydi ve orada SPLY tamamen boş kalıyor — özelliği
eğitimde hiç görmeyen model onu test'te de kullanamaz.

Bu yüzden bu denemeyi **Ocak-Mart 2026** üzerinde yapıyoruz: SPLY'nin (Ocak-Mart 2025)
mevcut olduğu tek dönem burası.

In [4]:
veri = veri_seti(train[train['tarih'] >= '2026-01-01'], pd.Timestamp('2025-12-01'))
veri['hedef_log'] = np.log1p(veri['tuketim_duzeltilmis'])

print('satır:', len(veri))
print('SPLY dolu olan oran: %.1f%%' % (veri['sply_w7'].notna().mean()*100))

satır: 338187
SPLY dolu olan oran: 46.3%


## 1. Sinyal var mı?

Önce korelasyonlara bakalım. Ama asıl önemli olan **kısmi korelasyon**: SPLY'nin
mevcut özelliklerin (trafonun genel seviyesi, son 30 günü) açıkladığının ötesinde
bir katkısı var mı? Çünkü iki değişken de aynı şeyi ölçüyorsa, ikincisi işe yaramaz.

In [5]:
d = veri[veri['sply_w7'].notna()]

for w in PENCERELER:
    print(f'corr(hedef, sply_w{w:<2}) = {d["hedef_log"].corr(d[f"sply_w{w}"]):.4f}')
print()
print(f'corr(hedef, log_ortalama)  = {d["hedef_log"].corr(d["log_ortalama"]):.4f}')
print(f'corr(hedef, son30_log_ort) = {d["hedef_log"].corr(d["son30_log_ort"]):.4f}')

# Kısmi korelasyon: log_ortalama ve son30'un etkisini ikisinden de çıkarıp kalana bakıyoruz
alt = d[['hedef_log','sply_w7','log_ortalama','son30_log_ort']].dropna()
X = np.c_[np.ones(len(alt)), alt['log_ortalama'], alt['son30_log_ort']]
artik_hedef = alt['hedef_log'] - X @ np.linalg.lstsq(X, alt['hedef_log'], rcond=None)[0]
artik_sply  = alt['sply_w7']   - X @ np.linalg.lstsq(X, alt['sply_w7'],  rcond=None)[0]
print()
print(f'kısmi korelasyon = {np.corrcoef(artik_hedef, artik_sply)[0,1]:.4f}')

corr(hedef, sply_w0 ) = 0.8126
corr(hedef, sply_w3 ) = 0.8171
corr(hedef, sply_w7 ) = 0.8181
corr(hedef, sply_w15) = 0.8209

corr(hedef, log_ortalama)  = 0.8737
corr(hedef, son30_log_ort) = 0.9291

kısmi korelasyon = 0.0381


## 2. Modelde ölçüm

Ocak-Şubat'ta eğitip Mart'ta ölçüyoruz. İki skor veriyoruz: tüm satırlar ve
sadece SPLY'nin dolu olduğu satırlar. İkincisi önemli, çünkü özellik satırların
yarısında boş olduğu için genel skorda etkisi sulanıyor.

In [6]:
egitim = veri[veri['tarih'] <= '2026-02-28'].copy()
dogrulama = veri[veri['tarih'] >= '2026-03-01'].copy()
for df in [egitim, dogrulama]:
    for c_ in KATEGORIK:
        df[c_] = df[c_].astype('category')

dolu = dogrulama['sply_w7'].notna()


def dene(ozellikler, etiket):
    m = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.05, num_leaves=63,
                          random_state=42, verbose=-1)
    m.fit(egitim[ozellikler], egitim['hedef_log'],
          eval_set=[(dogrulama[ozellikler], dogrulama['hedef_log'])], eval_metric='rmse',
          categorical_feature=KATEGORIK, callbacks=[lgb.early_stopping(80, verbose=False)])
    t = np.clip(np.expm1(m.predict(dogrulama[ozellikler])), 0, None)
    print(f'{etiket:26s} tümü={rmsle(dogrulama["tuketim_duzeltilmis"], t):.4f}   '
          f'SPLY-dolu={rmsle(dogrulama.loc[dolu,"tuketim_duzeltilmis"], t[dolu.values]):.4f}')


dene(TEMEL, 'referans')
for w in PENCERELER:
    dene(TEMEL + [f'sply_w{w}'], f'+ sply_w{w}')
dene(TEMEL + ['sply_sapma'], '+ sply_sapma')
dene(TEMEL + ['sply_w7','sply_sapma','sply_var'], '+ w7 + sapma + bayrak')

/Users/beratmertkayacan/Desktop/GridUpDatathon/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


referans                   tümü=0.8296   SPLY-dolu=0.4658


/Users/beratmertkayacan/Desktop/GridUpDatathon/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


+ sply_w0                  tümü=0.8280   SPLY-dolu=0.4562


/Users/beratmertkayacan/Desktop/GridUpDatathon/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


...

+ sply_w15                 tümü=0.8275   SPLY-dolu=0.4577


/Users/beratmertkayacan/Desktop/GridUpDatathon/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


+ sply_sapma               tümü=0.8408   SPLY-dolu=0.4925


/Users/beratmertkayacan/Desktop/GridUpDatathon/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


+ w7 + sapma + bayrak      tümü=0.8335   SPLY-dolu=0.4883


## Sonuç

Korelasyon tek başına yüksek (~0.82) ama **kısmi korelasyon ~0.04**. Yani SPLY'nin
taşıdığı bilginin neredeyse tamamı, trafonun genel seviyesi ve son 30 günü tarafından
zaten sağlanıyor. Modelde de kazanç ölçülebilir düzeyde değil (~0.003, iterasyon
sayısındaki oynaklığın altında).

Pencere karşılaştırmasında da belirgin bir kazanan yok — ±7 gün ile tam aynı gün
arasında anlamlı fark çıkmadı.

**Sebebi:** Trafo tüketimi büyük ölçüde kalıcı bir seviye etrafında dönüyor ve o
seviyeyi en iyi *en yakın* geçmiş anlatıyor, bir yıl öncesi değil. Yıllık mevsimsel
tekrar trafo bazında gürültülü.

**Bir çekince:** Bu ölçüm kış üzerinde (Ocak-Mart). Mevsimsel oynaklığın çok daha
büyük olduğu yazda (Temmuz medyanı Mayıs'ın 2 katı) SPLY daha değerli olabilir.
Ama veri 15 aylık olduğu için yaz üzerinde aynı testi kurmak mümkün değil —
Nisan-Temmuz 2025'in "geçen yılı" veride yok.

**Karar:** Ana pipeline'a eklenmiyor. Ölçülebilir katkısı yok ve yaz için doğrulanamıyor.